In [15]:
import numpy as np
import pandas as pd
from scipy.stats import pearsonr, spearmanr
from scipy.spatial.distance import jensenshannon
from sklearn.metrics import mean_squared_error
from skimage.metrics import structural_similarity as ssim
from libpysal.weights import KNN
from esda.moran import Moran
import numpy as np

def moran_score(pred, true):
    pred = np.asarray(pred, dtype=float)
    true = np.asarray(true, dtype=float)
    # 避免除零：当 true == 0 时，如果 pred == 0 返回1，否则返回0
    # 但按照原公式，如果 true == 0 且 pred <= true (即 pred <= 0)，只有 pred=0 时成立，0/0 是 nan。
    # 我们可以计算两个分支，然后选择。
    with np.errstate(divide='ignore', invalid='ignore'):
        branch1 = pred / true
        branch2 = (1 - pred) / (1 - true)
    result = np.where(pred <= true, branch1, branch2)
    # 处理 true == 0 的情况
    if np.any(true == 0):
        # 对于 true == 0 且 pred == 0，结果应为 1
        # 对于 true == 0 且 pred > 0，原公式会走 branch2，得到 (1-pred)/(1-0) = 1-pred，这不太合理。
        # 但通常 true 不会为0，我们可以额外处理使结果在 [0,1] 且符合直觉。
        # 为了简单，我们仅处理 pred=0, true=0 -> 1，其他 true=0 -> 0
        zero_true_mask = (true == 0)
        result = np.where(zero_true_mask & (pred == 0), 1.0, result)
        result = np.where(zero_true_mask & (pred != 0), 0.0, result)
    return result

def evaluate_deconvolution(pred, true, coords=None, eps=1e-12,
                           ssim_grid_size=64, knn_k=6):
    """
    Evaluate deconvolution prediction against ground truth.

    Parameters:
        pred : np.ndarray, shape (n_spots, n_types)
        true : np.ndarray, shape (n_spots, n_types)
        coords : np.ndarray, shape (n_spots, 2), optional
            Spatial coordinates of spots. Required for Moran's I and SSIM.
        eps : float
        ssim_grid_size : int
            Grid size for rasterizing spatial maps (e.g., 64 -> 64x64).
        knn_k : int
            Number of neighbors for Moran's I spatial weight matrix.

    Returns:
        pd.DataFrame with all metrics.
    """
    pred = np.asarray(pred)
    true = np.asarray(true)
    assert pred.shape == true.shape, "Shape mismatch"
    n_spots, n_types = pred.shape

    # ---- 1. Per-spot PCC / Spearman ----
    spot_pcc, spot_spearman = [], []
    for i in range(n_spots):
        p, t = pred[i, :], true[i, :]
        if np.std(p) > eps and np.std(t) > eps:
            pcc, _ = pearsonr(p, t)
            spearman, _ = spearmanr(p, t)
            spot_pcc.append(pcc)
            spot_spearman.append(spearman)
    mean_spot_pcc = np.nanmean(spot_pcc) if spot_pcc else np.nan
    mean_spot_spearman = np.nanmean(spot_spearman) if spot_spearman else np.nan

    # ---- 2. Per-cell-type PCC / Spearman ----
    type_pcc, type_spearman = [], []
    for j in range(n_types):
        p, t = pred[:, j], true[:, j]
        if np.std(p) > eps and np.std(t) > eps:
            pcc, _ = pearsonr(p, t)
            spearman, _ = spearmanr(p, t)
            type_pcc.append(pcc)
            type_spearman.append(spearman)
    mean_type_pcc = np.nanmean(type_pcc) if type_pcc else np.nan
    mean_type_spearman = np.nanmean(type_spearman) if type_spearman else np.nan

    # ---- 3. MSE / RMSE ----
    mse = mean_squared_error(true, pred)
    rmse = np.sqrt(mse)

    # ---- 4. Jensen-Shannon Divergence (per spot) ----
    jsd_list = []
    for i in range(n_spots):
        p = np.maximum(pred[i, :], 0)
        t = np.maximum(true[i, :], 0)
        if np.sum(p) < eps or np.sum(t) < eps:
            continue
        p = p / (np.sum(p) + eps)
        t = t / (np.sum(t) + eps)
        jsd = jensenshannon(p, t, base=2) ** 2
        jsd_list.append(jsd)
    mean_jsd = np.mean(jsd_list) if jsd_list else np.nan

    # ---- 5. Spatial metrics (SSIM + Moran's I) ----
    mean_ssim, mean_moran_pred, mean_moran_true, mean_moran_diff = np.nan, np.nan, np.nan, np.nan
    if coords is not None:
        coords = np.asarray(coords)

        # 5a. SSIM per cell type (rasterize spatial map to a 2D grid)
        ssim_list = []
        for j in range(n_types):
            pred_grid = _rasterize(coords, pred[:, j], grid_size=ssim_grid_size)
            true_grid = _rasterize(coords, true[:, j], grid_size=ssim_grid_size)
            # Skip if either grid is constant (SSIM undefined)
            if np.std(pred_grid) < eps or np.std(true_grid) < eps:
                continue
            # Use a fixed data range for both
            data_range = max(pred_grid.max(), true_grid.max()) - min(pred_grid.min(), true_grid.min())
            if data_range < eps:
                continue
            s = ssim(pred_grid, true_grid, data_range=data_range)
            ssim_list.append(s)
        mean_ssim = np.nanmean(ssim_list) if ssim_list else np.nan

        # 5b. Moran's I per cell type
        try:
            w = KNN.from_array(coords, k=knn_k)
            w.transform = 'r'  # row-standardized
            moran_pred_list, moran_true_list = [], []
            for j in range(n_types):
                p = pred[:, j]
                t = true[:, j]
                if np.std(p) > eps:
                    mi_p = Moran(p, w).I
                    moran_pred_list.append(mi_p)
                if np.std(t) > eps:
                    mi_t = Moran(t, w).I
                    moran_true_list.append(mi_t)
            mean_moran_pred = np.nanmean(moran_pred_list) if moran_pred_list else np.nan
            mean_moran_true = np.nanmean(moran_true_list) if moran_true_list else np.nan
            if not np.isnan(mean_moran_pred) and not np.isnan(mean_moran_true):
                mean_moran_diff = moran_score(mean_moran_pred , mean_moran_true)
        except Exception as e:
            print(f"[Warning] Moran's I failed: {e}")

    # ---- Assemble results ----
    results = {
        'metric': [
            'mean_spot_pcc', 'mean_spot_spearman',
            'mean_type_pcc', 'mean_type_spearman',
            'overall_mse', 'overall_rmse', 'mean_jsd',
            'mean_ssim', 'mean_moran_pred', 'mean_moran_true', 'mean_moran_diff'
        ],
        'value': [
            mean_spot_pcc, mean_spot_spearman,
            mean_type_pcc, mean_type_spearman,
            mse, rmse, mean_jsd,
            mean_ssim, mean_moran_pred, mean_moran_true, mean_moran_diff
        ]
    }
    df = pd.DataFrame(results).set_index('metric')
    return df


def _rasterize(coords, values, grid_size=64):
    """
    Rasterize scattered spatial values onto a 2D grid using nearest-neighbor assignment.
    Empty grid cells are filled with 0.
    """
    coords = np.asarray(coords)
    values = np.asarray(values)
    x, y = coords[:, 0], coords[:, 1]
    x_norm = (x - x.min()) / (x.max() - x.min() + 1e-12)
    y_norm = (y - y.min()) / (y.max() - y.min() + 1e-12)
    xi = np.clip((x_norm * (grid_size - 1)).astype(int), 0, grid_size - 1)
    yi = np.clip((y_norm * (grid_size - 1)).astype(int), 0, grid_size - 1)

    grid = np.zeros((grid_size, grid_size), dtype=float)
    count = np.zeros((grid_size, grid_size), dtype=float)
    np.add.at(grid, (yi, xi), values)
    np.add.at(count, (yi, xi), 1)
    # Average per grid cell
    grid = np.divide(grid, count, out=np.zeros_like(grid), where=count > 0)
    return grid

In [16]:
#ours
import scanpy as sc
import pandas as pd
A_gt_deconv=pd.read_csv('/home/qyyuan/project/ST_CP/JE/compare/groundtruth/NM_yangjy_X/data/S3_GT.txt',sep='\t',header=0,index_col=0)


In [17]:
pred_ours = np.load('/home/qyyuan/project/ST_CP/JE/NC_review/Methods/A_optimized_pca_gen_PCC_0_0.1_200.npy')
adata_sc = sc.read('/home/qyyuan/project/ST_CP/JE/method/data/scRNA_subsampled_50k_cleaned.h5ad')
adata_st = sc.read("/home/qyyuan/project/ST_CP/JE/compare/groundtruth/NM_yangjy_X/data/Visium_FAD.h5ad")
spot_clean = pd.DataFrame(adata_st.obsm['spatial'],index=adata_st.obs_names)
pred_ours=pred_ours/pred_ours.sum(axis=1,keepdims=True)
pred_ours[pred_ours<0.045]=0
pred_ours=pred_ours/pred_ours.sum(axis=1,keepdims=True)
df_results_ours = evaluate_deconvolution(pred_ours, A_gt_deconv.values,coords=spot_clean.loc[A_gt_deconv.index])

In [18]:
df_results_ours

,value
metric,
mean_spot_pcc,0.567998
mean_spot_spearman,0.571222
mean_type_pcc,0.275151
mean_type_spearman,0.252364
overall_mse,0.008379
overall_rmse,0.091537
mean_jsd,0.35261
mean_ssim,0.555903
mean_moran_pred,0.711834


In [19]:
# novosparc
A_pred=np.load("/home/qyyuan/project/ST_CP/JE/compare/novosparc/A_alpha_1_jiyanyang-alpha1-epsilon-3.npy")
A_pred_df=pd.DataFrame(A_pred,index=adata_sc.obs_names,columns=adata_st.obs_names)
A_pred_df=A_pred_df[A_gt_deconv.index]
celltype_label=pd.read_csv('/home/qyyuan/project/ST_CP/JE/method/data/scRNA_subsampled_50k_cleaned_metadata.csv',sep=',',index_col=0)
from sklearn.preprocessing import OneHotEncoder
ct_onehot_df = pd.get_dummies(celltype_label['cell_type'])  # 
# 2
spot_ct_proportions = A_pred_df.T @ ct_onehot_df  # n_spots × n_types
row_sums = spot_ct_proportions.sum(axis=1)  # Series
spot_ct_proportions = spot_ct_proportions.div(row_sums, axis=0)  # 按
spot_ct_proportions=spot_ct_proportions[A_gt_deconv.columns]
pred = spot_ct_proportions.values.astype(np.float64)
df_results_novosparc = evaluate_deconvolution(pred, A_gt_deconv.values,coords=spot_clean.loc[A_gt_deconv.index])

In [20]:
df_results_novosparc

,value
metric,
mean_spot_pcc,0.418778
mean_spot_spearman,0.420249
mean_type_pcc,0.09856
mean_type_spearman,0.103895
overall_mse,0.010193
overall_rmse,0.100959
mean_jsd,0.47806
mean_ssim,0.237672
mean_moran_pred,0.737633


In [21]:
## tangram
import scanpy as sc
pred_st=sc.read('/home/qyyuan/project/ST_CP/JE/Tangram/spatial_with_tangram-Version3-work.h5ad')
pred_st_df=pd.DataFrame(pred_st.X.toarray())
pred_st_df.index=pred_st.obs_names
pred_st_df.columns=pred_st.var_names
pred_A = sc.read('/home/qyyuan/project/ST_CP/JE/Tangram/tangram_mapping-Version3-work.h5ad')
spot_ct_proportions=pred_A.X.T @ ct_onehot_df
row_sums = spot_ct_proportions.sum(axis=1)  # Series
spot_ct_proportions = spot_ct_proportions.div(row_sums, axis=0) 
spot_ct_proportions=spot_ct_proportions[A_gt_deconv.columns]
spot_ct_proportions.index = pred_A.var_names
spot_ct_proportions=spot_ct_proportions.loc[A_gt_deconv.index]
pred = spot_ct_proportions.values.astype(np.float64)
df_results_tangram = evaluate_deconvolution(pred, A_gt_deconv.values,coords=spot_clean.loc[A_gt_deconv.index])

In [22]:
df_results_tangram

,value
metric,
mean_spot_pcc,0.42467
mean_spot_spearman,0.369378
mean_type_pcc,0.20924
mean_type_spearman,0.164334
overall_mse,0.010109
overall_rmse,0.100543
mean_jsd,0.457627
mean_ssim,0.27272
mean_moran_pred,0.297108


In [23]:
# RCTD
pred_A=pd.read_csv("/home/qyyuan/project/ST_CP/JE/compare/RCTD/deconv_jiyuanyang.txt",sep='\t',header=0,index_col=0)
# 将列名中的 '/' 替换为 ' ' (空格)
gtcol =A_gt_deconv.columns.str.replace('/', ' ', regex=False)
pred_A=pred_A[gtcol]
pred_A=pred_A[gtcol]
pred_A=pred_A.loc[A_gt_deconv.index]
pred_A = pred_A.div(pred_A.sum(axis=1), axis=0)
df_results_RCTD = evaluate_deconvolution(pred_A, A_gt_deconv.values,coords=spot_clean.loc[A_gt_deconv.index])

In [24]:
# CIBERSORTx
pred_A=pd.read_csv("/data/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/jiyuanYang/deconvolution/CIBERSORTx_MB.txt",sep='\t',header=0,index_col=0)
gtcol =A_gt_deconv.columns.str.replace('/', '_', regex=False).str.replace(' ', '_', regex=False)
pred_A=pred_A[gtcol]
pred_A=pred_A.loc[A_gt_deconv.index]
pred_A = pred_A.div(pred_A.sum(axis=1), axis=0)
df_results_CIBERSORTx = evaluate_deconvolution(pred_A, A_gt_deconv.values,coords=spot_clean.loc[A_gt_deconv.index])

In [25]:
#spotiphy
A_dev_pred=pd.read_csv('/home/qyyuan/project/ST_CP/JE/compare/spotiphy/Spotiphy_Result/proportion.csv',index_col=0)
A_dev_pred=A_dev_pred.loc[A_gt_deconv.index]
A_dev_pred=A_dev_pred[A_gt_deconv.columns]
df_results_spotiphy = evaluate_deconvolution(A_dev_pred.values, A_gt_deconv.values,coords=spot_clean.loc[A_gt_deconv.index])

In [26]:
# cell2location
A_dev_pred=pd.read_csv('/home/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/jiyuanYang/deconvolution/cell2location_results/q05_cell_abundance_w_sf.csv',index_col=0)
A_dev_pred.columns = A_dev_pred.columns.str.replace('q05cell_abundance_w_sf_', '')
A_dev_pred=A_dev_pred.loc[A_gt_deconv.index]
A_dev_pred=A_dev_pred[A_gt_deconv.columns]
A_dev_pred = A_dev_pred.div(A_dev_pred.sum(axis=1), axis=0)
df_results_cell2loc = evaluate_deconvolution(A_dev_pred.values, A_gt_deconv.values,coords=spot_clean.loc[A_gt_deconv.index])

In [27]:
df_results_cell2loc

,value
metric,
mean_spot_pcc,0.496655
mean_spot_spearman,0.429653
mean_type_pcc,0.281892
mean_type_spearman,0.226272
overall_mse,0.009113
overall_rmse,0.095462
mean_jsd,0.405032
mean_ssim,0.493032
mean_moran_pred,0.652667


In [28]:
# spatialDLWS
A_dev_pred=pd.read_csv('/home/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/jiyuanYang/deconvolution/spatialDWLS_dec.csv',sep=',',index_col=0)
A_dev_pred.columns = A_dev_pred.columns.str.replace('q05cell_abundance_w_sf_', '')
A_dev_pred=A_dev_pred.loc[A_gt_deconv.index]
A_dev_pred=A_dev_pred[A_gt_deconv.columns.str.replace('/', '_').str.replace(' ', '_')]
A_dev_pred = A_dev_pred.div(A_dev_pred.sum(axis=1), axis=0)
df_results_spatialDLWS = evaluate_deconvolution(A_dev_pred.values, A_gt_deconv.values,coords=spot_clean.loc[A_gt_deconv.index])

In [29]:
df_results_spatialDLWS

,value
metric,
mean_spot_pcc,0.501931
mean_spot_spearman,0.554454
mean_type_pcc,0.150394
mean_type_spearman,0.114392
overall_mse,0.009302
overall_rmse,0.096447
mean_jsd,0.376831
mean_ssim,0.436066
mean_moran_pred,0.343157


In [30]:
A_gt_deconv.columns.str.replace('/', '_').str.replace(' ', '_')

Index(['Astro', 'BCell', 'CA', 'DG', 'Endo', 'L2_3_IT_CTX', 'L4_IT_CTX',
       'L4_5_IT_CTX', 'L5_IT_CTX', 'L5_PT_CTX', 'L5_6_IT_CTX', 'L5_6_NP_CTX',
       'L6_CT_CTX', 'L6_IT_CTX', 'L6b_CTX', 'Lamp5', 'Macrophage', 'Microglia',
       'Neutrophil', 'Oligo', 'Pvalb', 'SUB', 'Sncg', 'Sst', 'Sst_Chodl',
       'TCell', 'Vip'],
      dtype='object')

In [31]:
A_dev_pred=pd.read_csv('/data/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/jiyuanYang/deconvolution/Cytospace/cytospace_results/fractional_abundances_by_spot.csv',sep=',',header=0,index_col=0)

A_dev_pred = A_dev_pred.reindex(
    columns=A_gt_deconv.columns.str.replace('/', '_').str.replace(' ', '_'),
    fill_value=0
)
#cytospace
#A_dev_pred=A_dev_pred[A_gt_deconv.columns.str.replace('/', '_').str.replace(' ', '_')]
row_sums = A_dev_pred.sum(axis=1)
A_dev_pred = A_dev_pred.div(row_sums,axis=0)
A_dev_pred = A_dev_pred.fillna(0)
df_results_cytospace = evaluate_deconvolution(A_dev_pred.loc[A_gt_deconv.index].values, A_gt_deconv.values,coords=spot_clean.loc[A_gt_deconv.index])

In [32]:
df_results_cytospace

,value
metric,
mean_spot_pcc,0.367767
mean_spot_spearman,0.39666
mean_type_pcc,0.140294
mean_type_spearman,0.122107
overall_mse,0.030902
overall_rmse,0.175789
mean_jsd,0.601816
mean_ssim,0.577455
mean_moran_pred,0.169424


In [39]:
df_all = pd.concat(
    [df_results_ours, df_results_novosparc, df_results_tangram,
     df_results_spotiphy, df_results_RCTD,df_results_cytospace, df_results_spatialDLWS,df_results_cell2loc,df_results_CIBERSORTx],
    axis=1,
    keys=['scMagNet', 'novoSpaRc', 'Tangram', 'Spotiphy', 'RCTD','CytoSPACE', 'SpatialDWLS','cell2location','CIBERSORTx']
)

In [ ]:
df_all.T.to_csv('/data/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/jiyuanYang/deconvolution/dec_result.txt',sep='\t')

: 

In [35]:
#SIMO

In [36]:
import pandas as pd
df = pd.read_csv('/home/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/jiyuanYang/SIMO/Location_map1.csv', index_col=0)
matrix = df.pivot(index='cell', columns='spot', values='value')
matrix = matrix.fillna(0)

celltype_label=pd.read_csv('/home/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/jiyuanYang/deconvolution/cell_type_labels.tsv',sep='\t',index_col=0)
celltype_label=celltype_label.loc[matrix.index]
ct_onehot_df = pd.get_dummies(celltype_label['CellType']) 


In [37]:
spot_ct_proportions = matrix.T @ ct_onehot_df  # n_spots × n_types
row_sums = spot_ct_proportions.sum(axis=1)  # Series
spot_ct_proportions = spot_ct_proportions.div(row_sums, axis=0)  # 按
spot_ct_proportions=spot_ct_proportions[A_gt_deconv.columns.str.replace('/', '_').str.replace(' ', '_')]
spot_ct_proportions=spot_ct_proportions.loc[A_gt_deconv.index]
pred = spot_ct_proportions.values.astype(np.float64)
df_results_SIMO = evaluate_deconvolution(pred, A_gt_deconv.values)

In [38]:
df_results_SIMO

,value
metric,
mean_spot_pcc,0.250207
mean_spot_spearman,0.294252
mean_type_pcc,0.107965
mean_type_spearman,0.109165
overall_mse,0.014104
overall_rmse,0.118759
mean_jsd,0.602855
mean_ssim,NaN
mean_moran_pred,NaN
